# Assignment 1:
Simulating an NMR (nuclear magnetic resonance) Spectrum from a real molecular Hamiltonian.

*In this assignment you will build the spin Hamiltonian of an actual molecule, compute the free-induction decay (FID), Fourier transform it, and compare your result with an experimental spectrum.*

## 1. Liquid-state NMR effective Hamiltonian

### 1.1 The physical setup

A liquid-state NMR experiment places a molecule in a strong, static magnetic field $B_0\hat{z}$ — typically 10–20 T, corresponding to a proton Larmor frequency of 400–900 MHz. The nuclear spins (we focus on $ ^1H$, spin-$\frac{1}{2}$) interact with this field and with each other. In solution, the molecule tumbles rapidly and isotropically on the timescale of the spin dynamics, and this averaging profoundly simplifies the Hamiltonian.

### 1.2 From the lab frame to the rotating frame

The bare lab-frame Hamiltonian for $n$ protons in the static field is

$$
H_{\text{lab}} = -\sum_i \gamma_i B_0\, I_i^z + H_{\text{couplings}}~,
$$

where $\gamma_i$ is the gyromagnetic ratio ($\gamma/2\pi \approx 42.577$ MHz/T for protons) and $I_i^z = \frac{1}{2}\sigma_i^z$ is the spin-$z$ operator. The Larmor frequency $\omega_0 = -\gamma B_0$ is hundreds of megahertz — fast and uninteresting for chemistry. We move to a **rotating frame** at the spectrometer reference frequency $\omega_{\text{ref}}$ by the transformation

$$
|\tilde\psi\rangle = e^{i\omega_{\text{ref}}t\sum_i I_i^z}|\psi\rangle~.
$$

In this frame, the fast Larmor precession is removed. What remains are only the *offsets* from the reference frequency — the chemical shifts — and the spin-spin couplings. The effective rotating-frame Hamiltonian (in units where $\hbar = 1$) is the one we actually work with:

$$
\boxed{
H = \underbrace{\sum_{i} \omega_i\, I_i^{z}}_{\text{chemical shift}} + \underbrace{\sum_{i<j} 2\pi J_{ij}\,\mathbf{I}_i\cdot\mathbf{I}_j}_{\text{scalar }J\text{-coupling}}
}
$$

with $\mathbf{I}_i\cdot\mathbf{I}_j = I_i^x I_j^x + I_i^y I_j^y + I_i^z I_j^z$.

### 1.3 Chemical shift

Protons in different chemical environments experience *slightly different* local fields because the surrounding electron cloud partially **shields** the nucleus from $B_0$. The effective field at nucleus $i$ is $B_i = B_0(1 - \sigma_i)$, where $\sigma_i \ll 1$ is the **shielding constant** set by the local electron density. In the rotating frame, this produces a site-dependent offset

$$
\omega_i = 2\pi\,\delta_i \times \nu_{\text{spec}}~,
$$

where $\delta_i$ is the **chemical shift** in parts per million (ppm) and $\nu_{\text{spec}}$ is the spectrometer frequency in MHz. For example, a proton at $\delta = 3.7$ ppm on a 500 MHz instrument sits at $\omega_i = 2\pi \times 1850$ rad/s in the rotating frame.

Because $\delta_i$ is defined relative to a reference compound, it is independent of magnet strength.

### 1.4 The scalar *J*-coupling term

Nuclei also interact with each other through an **indirect, electron-mediated coupling**. A proton's spin polarizes the electrons in the bonding orbitals, and that polarization is felt by a neighboring nucleus through one or more bonds. This is the **scalar (J) coupling** or **indirect spin-spin coupling**.

In free solution, the molecule tumbles isotropically. The **direct (through-space) dipolar coupling** — which is anisotropic and depends on the orientation of the internuclear vector relative to $B_0$ — averages to *zero*. The J-coupling survives because it is mediated by the isotropic electronic wave function: it is already rotationally invariant, giving a Heisenberg-type interaction

$$
H_J = \sum_{i<j} 2\pi J_{ij}\,\mathbf{I}_i\cdot\mathbf{I}_j = \sum_{i<j} 2\pi J_{ij}\left(I_i^x I_j^x + I_i^y I_j^y + I_i^z I_j^z\right).
$$

The coupling constant $J_{ij}$ is quoted in **Hz**, ranges from $\sim$0 to $\sim$20 Hz for vicinal proton couplings, and is tabulated for known molecules. The factor $2\pi$ converts to angular frequency to match the $\omega_i$ in the chemical shift term.

**Physical summary:** $H_J$ is an isotropic Heisenberg exchange between spin pairs, mediated by bonding electrons. It is what splits a bare resonance into a multiplet: without coupling, each proton gives one line; with coupling, the degeneracy is broken and a multiplet appears.

### 1.5 The weak- vs. strong-coupling limit

The ratio $|J_{ij}|/|\nu_i - \nu_j|$ governs which regime you are in:

- **Weak coupling** ($|J_{ij}| \ll |\nu_i - \nu_j|$): the flip-flop terms $I_i^+I_j^- + I_i^-I_j^+$ embedded in $I_i^xI_j^x + I_i^yI_j^y$ are far off-resonance and can be dropped. The coupling reduces to $2\pi J_{ij} I_i^z I_j^z$, which is diagonal in the product-$z$ basis. The familiar first-order rule ($n$ equivalent neighbors → $n+1$ lines with binomial intensities) follows algebraically, with no diagonalization needed.

- **Strong coupling** ($|J_{ij}| \sim |\nu_i - \nu_j|$): the flip-flop terms are resonant and mix product states. Eigenstates are no longer pure product states; line positions shift, intensities redistribute (the "roofing" effect, where inner lines of a multiplet grow at the expense of outer ones), and extra lines can appear. The first-order rules fail, and you must diagonalize $H$ exactly.

This is why NMR is an ideal ED problem: **the physically interesting regime is exactly where the simple analytical shortcut breaks down.**

### 1.6 What the Hamiltonian does *not* include

Two important effects are deliberately absent from $H$:

1. **Relaxation** ($T_1$, $T_2$): the slow return to thermal equilibrium and the dephasing of transverse magnetization. These involve the environment (molecular tumbling, solvent collisions) and cannot be written as a unitary evolution. This can be added phenomenologically as a multiplicative decay $e^{-t/T_2}$.

2. **The direct dipolar coupling**: it is real but averages to zero in isotropic solution. In solid-state NMR it is dominant; in liquid-state NMR it is negligible.

### 2. The Free Induction Decay and its spectrum

#### 2.1 The pulse–acquire experiment

The NMR experiment we simulate has three steps.

**Equilibration.** In the strong field $B_0\hat{z}$, the spins reach thermal equilibrium. At room temperature the nuclear Zeeman splitting is tiny compared to $k_BT$, so the equilibrium density matrix $\propto e^{\mu B/k_BT}|I_z=1/2\rangle \langle I_z=1/2| + e^{-\mu B/k_BT}|I_z=-1/2\rangle \langle I_z=-1/2|\approx \mathbb{1}_2 $ is nearly the identity. The small deviation from the identity is proportional to the total longitudinal magnetization:
$$\rho_{\text{eq}} \approx \frac{\mathbb{1}_n}{2^n} + \epsilon \sum_i I_i^z~,$$
where $\epsilon = \hbar\omega_0 / 2k_BT \ll 1$. Since the identity commutes with everything and produces no measurable signal, we track only the *deviation* $\rho_{\text{dev}} \propto \sum_i I_i^z$.

**The 90° pulse.** A short radio-frequency pulse rotates every spin by 90° about $\hat{y}$, mapping $I_i^z \to I_i^x$. The state immediately after the pulse is:
$$\rho(0) \propto I^x \equiv \sum_i I_i^x~.$$
This is the initial condition for the subsequent free evolution.

**Free evolution and detection.** The RF pulse is switched off. The spins evolve under $H$ and the transverse magnetization induces a voltage in the pickup coil. The coil is sensitive to the complex transverse magnetization, detected through the raising operator $I^+ = \sum_{j=1}^n I_j^+$, where $I^+_j = I_j^x + iI_j^y$. The measured complex signal, the **free induction decay** (FID), is:

$$
\boxed{s(t) = \mathrm{Tr}\!\left[I^+\, e^{-iHt}\, I^x\, e^{+iHt}\right] e^{-t/T_2}}
$$

The factor $e^{-t/T_2}$ is a phenomenological decay accounting for all dephasing mechanisms not in $H$ (molecular tumbling, field inhomogeneity, etc.). It turns perfectly sharp lines (delta functions) inteh spectrum into Lorentzians of half-width $1/\pi T_2$ Hz; in practice $T_2$ is matched to the experimental linewidth.

#### 2.2 The FID as a sum of oscillating exponentials

To see what $s(t)$ contains, insert the eigenbasis of $H$, with $H|a\rangle = E_a|a\rangle$:

$$
s(t) = \sum_{a,b} \underbrace{\langle b | I^+ | a \rangle\, \langle a | I^x | b \rangle}_{A_{ab}} \; e^{-i(E_a - E_b)\,t}\, e^{-t/T_2}~.
$$

The FID is a **superposition of damped complex exponentials**, one per pair of eigenstates $(a, b)$, each oscillating at the transition frequency $\omega_{ab} = E_a - E_b$ with a complex amplitude $A_{ab}$ set by the matrix elements of the preparation ($I^x$) and detection ($I^+$) operators.

Note that:

- **Amplitude:** $A_{ab} = 0$ unless both operators connect states $a$ and $b$. Because $\rho(0) \propto I^x$ and $I^+ = I^x + iI^y$, only transitions with nonzero spin-flip matrix elements contribute.
- **Selection rule:** $I^+$ raises the total magnetic quantum number $M = \sum_i m_i$ by exactly 1. Therefore $\langle b | I^+ | a\rangle \neq 0$ only when $M_b = M_a + 1$, i.e. only transitions of $\Delta M = \pm 1$ appear in the spectrum. 

#### 2.3 From FID to spectrum: the Fourier transform

Taking the Fourier transform of $s(t)$ converts the time-domain oscillations into frequency-domain peaks:

$$
S(\nu) = \int_0^\infty s(t)\, e^{+i 2\pi\nu t}\, dt = \sum_{a,b} A_{ab}\, \mathcal{L}(\nu - \nu_{ab})~,
$$

where $\nu_{ab} = (E_a - E_b)/2\pi$ and $\mathcal{L}(\nu) = \frac{T_2^{-1}/\pi}{\nu^2 + T_2^{-2}}$ is a Lorentzian of half-width $1/\pi T_2$ Hz. The real part of $S(\nu)$ is the **absorption spectrum**: a collection of Lorentzian peaks, one at each active transition frequency, with heights proportional to $|A_{ab}|$.

**What you see in a spectrum** is therefore exactly the set of eigenvalue *differences* of $H$, filtered through the selection rule and weighted by transition amplitudes. Chemists have been reading these differences as molecular fingerprints for 80 years.


## Problem: NMR spectrum of thymidine

As an application, we will calculate the NMR spectrum of thymidine, a naturally occurring pyrimidine deoxynucleoside in DNA (the 'T' in ATCG). https://en.wikipedia.org/wiki/Thymidine

Its chemical structure is shown below:

![Figure](50-89-5_1HNMR_2_0.jpg)

We are interested in the NMR spectrum from the H atoms labeled A through K in the figure, 11 in total. Each H atom nucleus is a proton, which has spin-1/2.

We will use the data from BMRB entry bmse000244 (https://bmrb.io/metabolomics/mol_summary/show_data.php?id=bmse000244), with the experimental condition D₂O, 400 MHz, pH 7.4, 298 K.

The following code cell records the chemical shifts and coupling constants $J_{ij}$. Use the provided data to

1. (20 pts) Construct the NMR spin Hamiltonian. 
2. (20 pts) Calculate the FID and plot the result on a figure . 
3. (20 pts) Calculate the Fourier transform of the FID, obtaining the predicted NMR spectrum. Plot it in a way so that it can be easily compared with the experimental data in the following figure.
4. (20 pts, answer in a separate markdown cell.) Compare your result with the experimental data at https://bmrb.io/metabolomics/mol_summary/show_data.php?id=bmse000244
Answer the following questions: Does your prediction capture all the spectrum peaks? What discrepancies do you observe? What could be the reasons for the discrepancies between your calculated result and the experimental result?

![Figure](1H.png)



In [1]:
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# Thymidine spin-system parameters
#
# Chemical shifts : BMRB entry bmse000244
#                   D₂O, 400 MHz, pH 7.4, 298 K
#
# Mapping of conventional NMR names to BMRB Atom IDs (copied from bmse000244):
#
#   i   Conventional   BMRB Atom ID   δ (ppm)   Assignment
#   0        H6            H23         7.645     thymine C5=C6 vinyl H
#   1        H1'           H28         6.285     deoxyribose anomeric H
#   2        H2'a          H21         2.368  \  deoxyribose C2' (diastereotopic;
#   3        H2'b          H22         2.368  /  accidentally degenerate at 400 MHz)
#   4        H3'           H26         4.464     deoxyribose C3'
#   5        H4'           H27         4.017     deoxyribose C4'
#   6        H5'a          H24         3.798  \  deoxyribose C5' (diastereotopic;
#   7        H5'b          H25         3.798  /  accidentally degenerate at 400 MHz)
#   8        Me_a          H18         1.886  \
#   9        Me_b          H19         1.886   > thymine 5-methyl (magnetically equivalent)
#  10        Me_c          H20         1.886  /
#
# J-coupling constants : literature values for deoxyribose nucleosides
#   (Wijmenga & van Buuren, Prog. NMR Spectrosc. 32, 287–387, 1998)
# ─────────────────────────────────────────────────────────────────────────────

n_spins = 11

# Conventional NMR labels (used throughout the code)
labels = ["H6", "H1'", "H2'a", "H2'b", "H3'", "H4'", "H5'a", "H5'b",
          "Me_a", "Me_b", "Me_c"]

# Corresponding BMRB Atom IDs from bmse000244 (for traceability)
bmrb_ids = ["H23", "H28", "H21", "H22", "H26", "H27", "H24", "H25",
            "H18", "H19", "H20"]

nu_spec = 400.0   # spectrometer ¹H frequency (MHz)

# --- Chemical shifts δ_i (ppm) — from BMRB bmse000244 -----------------------
delta_ppm = np.array([
    7.645,   # H6    (BMRB H23)
    6.285,   # H1'   (BMRB H28)
    2.368,   # H2'a  (BMRB H21)  degenerate with H2'b at 400 MHz
    2.368,   # H2'b  (BMRB H22)
    4.464,   # H3'   (BMRB H26)
    4.017,   # H4'   (BMRB H27)
    3.798,   # H5'a  (BMRB H24)  degenerate with H5'b at 400 MHz
    3.798,   # H5'b  (BMRB H25)
    1.886,   # Me_a  (BMRB H18)
    1.886,   # Me_b  (BMRB H19)
    1.886,   # Me_c  (BMRB H20)
])

# Offset frequencies ν_i = δ_i × ν_spec (Hz); Hamiltonian uses H_shift = Σ_i 2π ν_i I_i^z
nu_Hz = delta_ppm * nu_spec

# --- J-coupling matrix J[i,j] (Hz) — symmetric, J[i,i] = 0 ------------------
J = np.zeros((n_spins, n_spins))

# H6 ↔ each 5-methyl proton  (4-bond long-range, ~1.3 Hz)
for me in [8, 9, 10]:
    J[0, me] = J[me, 0] = 1.3

# H1' ↔ H2'a, H2'b  (³J vicinal; both ≈ 6.6 Hz → apparent triplet for H1')
J[1, 2] = J[2, 1] = 6.6   # H1'–H2'a
J[1, 3] = J[3, 1] = 6.6   # H1'–H2'b

# H2'a ↔ H2'b  (²J geminal)
J[2, 3] = J[3, 2] = 13.7

# H2' ↔ H3'  (³J vicinal; unequal because dihedral angles differ)
J[2, 4] = J[4, 2] = 2.5   # H2'a–H3'
J[3, 4] = J[4, 3] = 6.4   # H2'b–H3'

# H3' ↔ H4'  (³J vicinal)
J[4, 5] = J[5, 4] = 3.2

# H4' ↔ H5'a, H5'b  (³J vicinal)
J[5, 6] = J[6, 5] = 3.2   # H4'–H5'a
J[5, 7] = J[7, 5] = 3.6   # H4'–H5'b

# H5'a ↔ H5'b  (²J geminal)
J[6, 7] = J[7, 6] = 11.5

# Me_a/b/c are magnetically equivalent; their mutual couplings do not affect
# the observable spectrum and are set to zero.

# --- Sanity checks -----------------------------------------------------------
assert np.allclose(J, J.T), "J must be symmetric"
assert np.all(np.diag(J) == 0)

# --- Summary printout --------------------------------------------------------
print(f"Molecule : thymidine  |  D₂O, {nu_spec:.0f} MHz, pH 7.4, 298 K")
print(f"n_spins = {n_spins}  →  Hilbert-space dim = 2^{n_spins} = {2**n_spins}\n")

print(f"{'i':>3}  {'label':>6}  {'BMRB ID':>7}  {'δ (ppm)':>9}  {'ν (Hz)':>9}")
print("─" * 44)
for i in range(n_spins):
    print(f"{i:>3}  {labels[i]:>6}  {bmrb_ids[i]:>7}  {delta_ppm[i]:>9.3f}  {nu_Hz[i]:>9.1f}")

print("\nNon-zero J-couplings (Hz):")
for i in range(n_spins):
    for j in range(i + 1, n_spins):
        if J[i, j] != 0.0:
            print(f"  J({labels[i]:>5}, {labels[j]:>5}) = {J[i,j]:5.1f} Hz")

Molecule : thymidine  |  D₂O, 400 MHz, pH 7.4, 298 K
n_spins = 11  →  Hilbert-space dim = 2^11 = 2048

  i   label  BMRB ID    δ (ppm)     ν (Hz)
────────────────────────────────────────────
  0      H6      H23      7.645     3058.0
  1     H1'      H28      6.285     2514.0
  2    H2'a      H21      2.368      947.2
  3    H2'b      H22      2.368      947.2
  4     H3'      H26      4.464     1785.6
  5     H4'      H27      4.017     1606.8
  6    H5'a      H24      3.798     1519.2
  7    H5'b      H25      3.798     1519.2
  8    Me_a      H18      1.886      754.4
  9    Me_b      H19      1.886      754.4
 10    Me_c      H20      1.886      754.4

Non-zero J-couplings (Hz):
  J(   H6,  Me_a) =   1.3 Hz
  J(   H6,  Me_b) =   1.3 Hz
  J(   H6,  Me_c) =   1.3 Hz
  J(  H1',  H2'a) =   6.6 Hz
  J(  H1',  H2'b) =   6.6 Hz
  J( H2'a,  H2'b) =  13.7 Hz
  J( H2'a,   H3') =   2.5 Hz
  J( H2'b,   H3') =   6.4 Hz
  J(  H3',   H4') =   3.2 Hz
  J(  H4',  H5'a) =   3.2 Hz
  J(  H4',  H5'b) 